# E01 Matrix Sensing Benchmark — PyTorch Notebook Rewrite

This notebook keeps the experiment logic visible: target matrix generation, measurement matrix generation, loss, run loop, metrics, plots, and CSV writing are all defined here. Only the stateful Muon optimizer is imported from `muonlib_torch`.

Default mode is a fast smoke run. Set `SMOKE_MODE = False` in the parameter cell for the E01-sized experiment.

In [ ]:
from pathlib import Path
import csv
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

PROJECT = Path.cwd().resolve()
if not (PROJECT / "muonlib_torch").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from muonlib_torch import MuonTorch

torch.set_default_dtype(torch.float64)
print(f"project={PROJECT}")
print(f"torch={torch.__version__}")

## Parameters

The experiment grid is intentionally explicit. Change these constants directly in the notebook.

In [ ]:
SMOKE_MODE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

ALGOS = ["Muon-Exact", "SGD"]
DIMS = [30] if SMOKE_MODE else [50, 60, 70]
R = 5
LR = 0.01
NOISE = 0.0
DIST = "normal"
SPECTRUM = "hard-cutoff"
KAPPA = 1.0
INIT_SCALE = 0.01
ITERS = 120 if SMOKE_MODE else 2000
SEEDS = list(range(2)) if SMOKE_MODE else list(range(10))
EPSILON = 0.01
PLOT_EVERY = 10 if SMOKE_MODE else 100

RESULTS_DIR = PROJECT / "results_torch"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE}, dtype={DTYPE}, smoke={SMOKE_MODE}")
print(f"runs={len(ALGOS) * len(DIMS) * len(SEEDS)}, iters={ITERS}")

## Matrix Sensing Math

Target matrix:

$$
X^\star = U \operatorname{diag}(s) V^\top
$$

Measurements:

$$
y_i = \langle A_i, X^\star\rangle + \varepsilon_i
$$

Loss:

$$
f(X) = \frac{1}{2m}\sum_{i=1}^{m}(\langle A_i, X\rangle-y_i)^2
$$

In [ ]:
def make_generator(seed):
    try:
        return torch.Generator(device=DEVICE).manual_seed(int(seed))
    except Exception:
        return torch.Generator().manual_seed(int(seed))


def randn(shape, seed):
    return torch.randn(shape, generator=make_generator(seed), device=DEVICE, dtype=DTYPE)


def generate_target_matrix(d, r, spectrum="hard-cutoff", kappa=1.0, seed=0):
    U, _ = torch.linalg.qr(randn((d, d), seed))
    V, _ = torch.linalg.qr(randn((d, d), seed + 17))

    s = torch.zeros(d, device=DEVICE, dtype=DTYPE)
    if spectrum == "hard-cutoff":
        s[:r] = 1.0
    elif spectrum == "polynomial-decay":
        idx = torch.arange(1, r + 1, device=DEVICE, dtype=DTYPE)
        s[:r] = idx.pow(-1.0)
        s[:r] /= s[0]
    elif spectrum == "exponential-decay":
        idx = torch.arange(r, device=DEVICE, dtype=DTYPE)
        s[:r] = torch.exp(-0.5 * idx)
    else:
        raise ValueError(f"unknown spectrum: {spectrum}")

    if kappa > 1.0 and r > 1:
        s[:r] = torch.linspace(1.0, 1.0 / kappa, r, device=DEVICE, dtype=DTYPE)

    return U @ torch.diag(s) @ V.T


def generate_measurements(d, m, dist="normal", seed=0, sparsity=0.1):
    g = make_generator(seed)
    if dist == "normal":
        A = torch.randn((m, d, d), generator=g, device=DEVICE, dtype=DTYPE)
    elif dist == "uniform":
        A = 2.0 * torch.rand((m, d, d), generator=g, device=DEVICE, dtype=DTYPE) - 1.0
    elif dist == "rademacher":
        A = torch.randint(0, 2, (m, d, d), generator=g, device=DEVICE).to(DTYPE)
        A = A.mul(2.0).sub(1.0)
    elif dist == "sparse":
        dense = torch.randn((m, d, d), generator=g, device=DEVICE, dtype=DTYPE)
        mask = torch.rand((m, d, d), generator=g, device=DEVICE, dtype=DTYPE) < sparsity
        A = dense * mask / (sparsity * d * d) ** 0.5
    elif dist == "sphere":
        A = torch.randn((m, d, d), generator=g, device=DEVICE, dtype=DTYPE)
        A = A / A.flatten(1).norm(dim=1).clamp_min(1e-12).view(m, 1, 1)
    else:
        raise ValueError(f"unknown measurement distribution: {dist}")
    return A


def matrix_sensing_loss(A, y, X):
    pred = torch.einsum("mij,ij->m", A, X)
    return 0.5 * torch.mean((pred - y) ** 2)

## Single Run

The run loop intentionally stays in the notebook. Note that it records `K_epsilon` but does not break early, matching the existing NumPy scripts.

In [ ]:
def make_optimizer(algo, params, lr):
    if algo == "Muon-Exact":
        return MuonTorch(params, lr=lr, momentum=0.9, variant="exact")
    if algo == "Muon-RandSVD":
        return MuonTorch(params, lr=lr, momentum=0.9, variant="randsvd", rank=R)
    if algo == "Muon-Trunc":
        return MuonTorch(params, lr=lr, momentum=0.9, variant="truncated", rank=R)
    if algo == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    if algo == "Adam":
        return torch.optim.Adam(params, lr=lr)
    raise ValueError(f"unknown algo: {algo}")


def maybe_sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def run_ms_once(algo, d, r, lr, noise, dist, spectrum, kappa, init_scale, seed, iters):
    X_star = generate_target_matrix(d, r, spectrum=spectrum, kappa=kappa, seed=seed)
    m = int(2 * d * r)
    A = generate_measurements(d, m, dist=dist, seed=seed + 1000)
    y = torch.einsum("mij,ij->m", A, X_star)
    if noise > 0:
        y = y + randn((m,), seed + 2000) * noise

    X0 = randn((d, d), seed + 3000) * init_scale
    X = torch.nn.Parameter(X0)
    opt = make_optimizer(algo, [X], lr)

    losses = []
    grad_norms = []
    k_epsilon = None
    maybe_sync()
    t0 = time.time()

    for step in range(iters):
        opt.zero_grad(set_to_none=True)
        loss = matrix_sensing_loss(A, y, X)
        loss.backward()

        grad_norm = float(X.grad.detach().norm().cpu())
        opt.step()

        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        grad_norms.append(grad_norm)

        if k_epsilon is None and loss_value <= EPSILON:
            k_epsilon = step + 1

    maybe_sync()
    elapsed = time.time() - t0
    if k_epsilon is None:
        k_epsilon = iters + 1

    row = {
        "algo": algo,
        "d": d,
        "r": r,
        "m_meas": m,
        "lr": lr,
        "noise": noise,
        "dist": dist,
        "spectrum": spectrum,
        "kappa": kappa,
        "init_scale": init_scale,
        "seed": seed,
        "iters": iters,
        "final_loss": losses[-1],
        "min_loss": min(losses),
        "K_epsilon": k_epsilon,
        "time_s": elapsed,
    }
    traj = {"loss": losses, "grad_norm": grad_norms}
    return row, traj

## Run Grid and Plot

Use `run_experiment(live=True)` directly, or click the button in the next cell if widgets are available.

In [ ]:
def plot_trajectories(trajectories):
    plt.figure(figsize=(8, 4.5))
    for key, traj in trajectories.items():
        algo, d, seed = key
        losses = traj["loss"]
        plt.plot(losses, alpha=0.75, label=f"{algo} d={d} seed={seed}")
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Matrix sensing loss trajectories")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_summary(df):
    if df.empty:
        return
    display(df.sort_values(["algo", "d", "seed"]))
    summary = df.groupby(["algo", "d"], as_index=False).agg(
        K_epsilon_mean=("K_epsilon", "mean"),
        min_loss_mean=("min_loss", "mean"),
        time_s_mean=("time_s", "mean"),
    )
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for algo in summary["algo"].unique():
        s = summary[summary["algo"] == algo]
        axes[0].plot(s["d"], s["K_epsilon_mean"], marker="o", label=algo)
        axes[1].plot(s["d"], s["time_s_mean"], marker="o", label=algo)
    axes[0].set_title("Mean K_epsilon")
    axes[0].set_xlabel("d")
    axes[0].set_ylabel("iterations")
    axes[1].set_title("Mean wall-clock")
    axes[1].set_xlabel("d")
    axes[1].set_ylabel("seconds")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def save_rows(rows):
    if not rows:
        return None
    path = RESULTS_DIR / "E01_ms_benchmark_torch_results.csv"
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return path


def run_experiment(live=False):
    rows = []
    trajectories = {}
    total = len(ALGOS) * len(DIMS) * len(SEEDS)
    idx = 0

    for algo in ALGOS:
        for d in DIMS:
            for seed in SEEDS:
                idx += 1
                print(f"[{idx}/{total}] {algo} d={d} seed={seed}", flush=True)
                row, traj = run_ms_once(
                    algo, d, R, LR, NOISE, DIST, SPECTRUM, KAPPA,
                    INIT_SCALE, seed, ITERS,
                )
                rows.append(row)
                trajectories[(algo, d, seed)] = traj

                if live:
                    clear_output(wait=True)
                    print(f"completed {idx}/{total}")
                    df_live = pd.DataFrame(rows)
                    plot_summary(df_live)
                    plot_trajectories(trajectories)

    df = pd.DataFrame(rows)
    result_path = save_rows(rows)
    print(f"saved {result_path}")
    return df, trajectories

## Click to Run

Click the button, or run `df, trajectories = run_experiment(live=True)` manually.

In [ ]:
if widgets is not None:
    run_button = widgets.Button(description="Run E01 torch", button_style="primary")
    out = widgets.Output()

    def on_run_clicked(_):
        with out:
            clear_output(wait=True)
            df, trajectories = run_experiment(live=True)
            clear_output(wait=True)
            plot_summary(df)
            plot_trajectories(trajectories)

    run_button.on_click(on_run_clicked)
    display(run_button, out)
else:
    print("ipywidgets is not installed. Run this manually:")
    print("df, trajectories = run_experiment(live=True)")